# revit-api-rag Pipeline

数据准备阶段 — 在 Colab 中运行

## 流程
1. 克隆项目 & 安装依赖
2. 上传原始数据（API HTML + SDK 代码）
3. 解析 API 文档 → SQLite
4. 解析 SDK 代码 → SQLite
5. Embedding → ChromaDB
6. 下载生成的 .db 文件

## 首次使用 — 按顺序运行所有 Cell

1. **环境准备** — 安装依赖、设置 API Key、配置路径
2. **解压数据** — CHM → api_html，ZIP → sdk_samples
3. **解析数据** — HTML → SQLite，.cs → SQLite
4. **Embedding** — SQLite → ChromaDB 向量库，打包到 Drive
5. **测试 RAG** — 检索 + LLM 生成

## 再次使用 — 只需运行标记为 🔄 的 Cell

1. 🔄 环境准备
2. 🔄 恢复数据（从 Drive 解压 tar.gz）
3. 🔄 测试 RAG

## Step 0: 环境准备

In [7]:
from google.colab import drive
drive.mount('/content/drive')
# 克隆项目
#!git clone https://github.com/imkcrevit/revit-api-rag.git
%cd /content/drive/MyDrive/Colab_Projects/revit-api-rag/

!ls

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Colab_Projects/revit-api-rag
config	legacy	  README.md		     scripts
data	LICENSE   requirements-pipeline.txt  server
docs	pipeline  requirements-server.txt


In [8]:
# 安装 pipeline 依赖
!pip install -r requirements-pipeline.txt -q

In [9]:
# 验证关键包是否安装成功
import chromadb
import google.genai
import yaml
print("所有关键依赖已就绪 ✅")

所有关键依赖已就绪 ✅


In [ ]:
%%bash
cat << 'EOF' > .env
OPENROUTER_API_KEY=KEYS
EOF

In [17]:
import os
from dotenv import load_dotenv

# 从项目根目录加载 .env 文件
load_dotenv(dotenv_path=".env")

api_key = os.getenv("OPENROUTER_API_KEY")

if not api_key:
    raise RuntimeError(
        "没有找到 OPENROUTER_API_KEY：\n"
        "- 请在项目根目录的 .env 文件中设置 OPENROUTER_API_KEY=你的 Key；\n"
        "- 或在系统环境变量中设置同名变量。"
    )

os.environ["OPENROUTER_API_KEY"] = api_key
print("API Key 已从 .env 设置 ✅")

API Key 已从 .env 设置 ✅


In [ ]:
# 在 Colab 中运行，看看现在的情况
%cd /content/drive/MyDrive/Colab_Projects/revit-api-rag

# 1. 把旧文件移到 legacy/ 目录保留（不删除，以后迁移代码时参考）
!mkdir -p legacy
!mv main.ipynb legacy/
!mv split_revit.ipynb legacy/
!mv selct_all_api.ipynb legacy/
!mv setup.py legacy/
!mv requirements.txt legacy/
!mv union_merge_config.yaml legacy/
!mv output_text_1023.md legacy/
!mv project_dataset.json legacy/
!mv full_api.txt legacy/
!mv deepseek_tokenizer_v3 legacy/
!mv extra_data legacy/
!mv revit_sdk_prund legacy/
!mv revit_sdk_collection legacy/

# 2. 旧数据库移到 data/ 目录（后续还要用）
!mkdir -p data/legacy_db
!mv chromadb0815_api_1.db data/legacy_db/
!mv chromadb1022_code_1.db data/legacy_db/
!mv revit_api.db data/legacy_db/

# 3. 旧图片移到 docs/
!mkdir -p docs/images
!mv *.png docs/images/
!mv *.jpg docs/images/

# 4. 把新骨架从嵌套目录提升到根目录
!cp -r revit-api-rag/pipeline ./
!cp -r revit-api-rag/server ./
!cp -r revit-api-rag/config ./
!cp -r revit-api-rag/scripts ./
!cp revit-api-rag/requirements-pipeline.txt ./
!cp revit-api-rag/requirements-server.txt ./
!cp revit-api-rag/README.md ./README.md

# 5. 删除嵌套目录和压缩包
!rm -rf revit-api-rag/
!rm -f revit-api-rag-skeleton.tar.gz

# 6. 创建数据目录
!mkdir -p data/{chromadb,sqlite,knowledge,raw}

# 7. 验证最终结构
!echo "=== 项目根目录 ==="
!ls
!echo ""
!echo "=== pipeline/ ==="
!ls pipeline/
!echo ""
!echo "=== server/ ==="
!ls server/
!echo ""
!echo "=== legacy/ ==="
!ls legacy/


In [19]:
%cd /content/drive/MyDrive/Colab_Projects/revit_data/

!ls

/content/drive/MyDrive/Colab_Projects/revit_data
api_html.tar.gz  RevitAPI.chm  Samples.zip  sdk_samples.tar.gz


In [20]:
# 复制配置文件
!cp config/config.example.yaml config/config.yaml
print('配置文件已创建 ✅')
print('如需修改 embedding provider，请编辑 config/config.yaml')

cp: cannot stat 'config/config.example.yaml': No such file or directory
配置文件已创建 ✅
如需修改 embedding provider，请编辑 config/config.yaml


## Step 1: 上传原始数据

In [ ]:
# 方式一：从 Google Drive 挂载
from google.colab import drive
drive.mount('/content/drive')

# 方式二：直接上传文件
# from google.colab import files
# uploaded = files.upload()

### [DEBUG] 解析质量全检（500 条抽样 → 字段统计 → QA Agent，不写库）

In [26]:
import sys, importlib, random, re
from pathlib import Path

# ── 参数 ─────────────────────────────────────────
HTML_DIR    = '/content/api_html/'   # CHM 解压目录
SAMPLE_N    = 500                    # 解析采样量
QA_SAMPLE_N = 50                     # QA Agent 采样量（节省 token）
RANDOM_SEED = 42
# ─────────────────────────────────────────────────

PROJECT_ROOT = '/content/drive/MyDrive/Colab_Projects/revit-api-rag'
sys.path.insert(0, PROJECT_ROOT)

# 确保拉取最新代码（含 quality_agent.py / llm_client.py 等新模块）
import subprocess
result = subprocess.run(['git', '-C', PROJECT_ROOT, 'pull', '--rebase'], capture_output=True, text=True)
print(result.stdout or result.stderr)

# 强制清空已缓存的旧模块，保证 reload 到最新版
for mod_name in list(sys.modules.keys()):
    if 'pipeline' in mod_name:
        del sys.modules[mod_name]

from pipeline.api_parser.parse_chm import parse_single_html
from pipeline.api_parser.quality_agent import run_quality_agent
import yaml

with open(f'{PROJECT_ROOT}/config/config.yaml') as f:
    config = yaml.safe_load(f)

# ── ctor 检测（内联，不依赖私有函数）─────────────
def _is_ctor(title: str, fid: str) -> bool:
    fid_low = fid.lower()
    if '.#ctor' in fid_low or '#ctor(' in fid_low or fid_low.endswith('#ctor'):
        return True
    if re.search(r'\bconstructors?\s*$', title, re.IGNORECASE):
        return True
    return False

# ════════════════════════════════════════════════
# 阶段 1：随机采样 & 解析
# ════════════════════════════════════════════════
all_files = list(Path(HTML_DIR).rglob('*.html')) + list(Path(HTML_DIR).rglob('*.htm'))
print(f'共发现 HTML 文件：{len(all_files)} 个')

random.seed(RANDOM_SEED)
sample_files = random.sample(all_files, min(SAMPLE_N, len(all_files)))

results = []
skipped_ctor = 0
skipped_noise = 0

for f in sample_files:
    data = parse_single_html(str(f))
    if data:
        results.append(data)
    else:
        # 区分 ctor 跳过和其他噪音
        try:
            from bs4 import BeautifulSoup
            html = Path(f).read_text(encoding='utf-8', errors='ignore')
            soup = BeautifulSoup(html, 'html.parser')
            t = soup.find('title')
            title = t.string.strip() if t and t.string else ''
            m = soup.find('meta', attrs={'name': 'Microsoft.Help.F1'})
            fid = m.get('content', '') if m else ''
            if _is_ctor(title, fid):
                skipped_ctor += 1
            else:
                skipped_noise += 1
        except Exception:
            skipped_noise += 1

print(f'\n采样 {len(sample_files)} 个文件 → 有效 {len(results)} 条')
print(f'  构造函数(ctor)过滤 : {skipped_ctor} 条')
print(f'  其他噪音页过滤     : {skipped_noise} 条\n')

# ── 字段完整性统计 ────────────────────────────────
fields = ['name', 'full_id', 'namespace', 'summary', 'info', 'parameters', 'syntax', 'members', 'remark']
print('=== 字段填充率 ===')
for field in fields:
    filled = sum(1 for r in results if r.get(field))
    pct = filled / len(results) * 100 if results else 0
    bar = '█' * int(pct / 5) + '░' * (20 - int(pct / 5))
    print(f'  {field:<14} {bar}  {filled:>4}/{len(results)}  ({pct:.1f}%)')

# ── 前 10 条详细内容 ──────────────────────────────
print('\n' + '='*70)
print('=== 前 10 条详细内容 ===')
print('='*70)

for i, item in enumerate(results[:10], 1):
    name     = item.get('name', '')
    full_id  = item.get('full_id', '')
    ns       = item.get('namespace', '')
    summary  = (item.get('summary') or item.get('info') or '')[:120]
    params   = item.get('parameters') or ''
    syntax   = (item.get('syntax') or '')[:80]
    member_n = len((item.get('members') or '').splitlines())

    print(f'\n[{i:02d}] {name}')
    print(f'     full_id   : {full_id}')
    print(f'     namespace : {ns}')
    print(f'     summary   : {summary or "（空）"}')
    if params:
        plines = params.strip().splitlines()
        print(f'     params({len(plines)}) :')
        for pl in plines[:3]:
            print(f'       {pl}')
        if len(plines) > 3:
            print(f'       ... (共 {len(plines)} 个)')
    if syntax:
        print(f'     syntax    : {syntax}...')
    if member_n:
        print(f'     members   : {member_n} 个成员')
    print('-'*70)

# ── 随机 5 条深度核查 ─────────────────────────────
print('\n=== 随机 5 条深度检查 ===')
for item in random.sample(results, min(5, len(results))):
    print(f'\n  [{item["name"]}]')
    print(f'  full_id : {item.get("full_id")}')
    print(f'  syntax  : {(item.get("syntax") or "")[:100]}')
    raw_p = item.get("parameters") or ""
    print(f'  params  : {raw_p[:200] if raw_p else "—"}')
    print(f'  summary : {(item.get("summary") or item.get("info") or "")[:150]}')

# ════════════════════════════════════════════════
# 阶段 2：Quality Agent 抽样审核（Gemini → Claude）
# ════════════════════════════════════════════════
print('\n\n' + '█'*70)
print('  Quality Agent 启动（Gemini 快速审核 + Claude Sonnet 兜底重写）')
print('█'*70 + '\n')

random.seed(RANDOM_SEED + 1)
qa_sample = random.sample(results, min(QA_SAMPLE_N, len(results)))

qa_results = run_quality_agent(
    qa_sample, config,
    html_dir=HTML_DIR,
    max_stage2=20,
    verbose=True,
)

# ── QA 结果展示 ───────────────────────────────────
low_q    = [r for r in qa_results if r['_quality_score'] < 0.6]
rewrites = [r for r in qa_results if r['_rewritten']]

print('\n' + '='*70)
print(f'=== 低质量条目（score < 0.6）共 {len(low_q)} 条 / {QA_SAMPLE_N} 条样本 ===')
print('='*70)

for item in low_q[:8]:
    print(f"\n  [{item['name']}]  score={item['_quality_score']:.2f}  重写={item['_rewritten']}")
    print(f"  issues  : {'; '.join(item['_quality_issues'][:3])}")
    print(f"  summary : {(item.get('summary') or item.get('info') or '')[:100]}")
    if item.get('parameters'):
        print(f"  params  : {item['parameters'][:120]}")
    print('-'*60)

print(f'\n被 Stage-2（Claude）重写共 {len(rewrites)} 条')


From https://github.com/imkcrevit/revit-api-rag
 * [new branch]      colab      -> origin/colab
 * [new branch]      main       -> origin/main
There is no tracking information for the current branch.
Please specify which branch you want to rebase against.
See git-pull(1) for details.

    git pull <remote> <branch>

If you wish to set tracking information for this branch you can do so with:

    git branch --set-upstream-to=origin/<branch> main




ImportError: cannot import name '_is_constructor_page' from 'pipeline.api_parser.parse_chm' (/content/drive/MyDrive/Colab_Projects/revit-api-rag/pipeline/api_parser/parse_chm.py)

In [25]:
# ── 已合并至上方调试 Block，此 Cell 可忽略 ──

print(f'对 {len(qa_sample)} 条抽样数据运行 Quality Agent...\n')
qa_results = run_quality_agent(qa_sample, config, html_dir=HTML_DIR, max_stage2=20, verbose=True)

# ── 展示低质量 + 被重写的条目 ─────────────────────
print('\n' + '='*70)
print('=== 低质量条目详情 ===')
print('='*70)

low_q = [r for r in qa_results if r['_quality_score'] < 0.6]
for item in low_q[:10]:
    print(f"\n  [{item['name']}]  score={item['_quality_score']:.2f}  rewritten={item['_rewritten']}")
    print(f"  issues : {item['_quality_issues']}")
    print(f"  summary: {(item.get('summary') or item.get('info') or '')[:120]}")
    if item.get('parameters'):
        print(f"  params : {item['parameters'][:150]}")
    print('-'*60)

print(f'\n低质量条目共 {len(low_q)} 条 / {len(qa_results)} 条样本')
rewrites = [r for r in qa_results if r['_rewritten']]
print(f'被 Stage-2 重写共 {len(rewrites)} 条')


ModuleNotFoundError: No module named 'pipeline.api_parser.quality_agent'

解压文件到目标位置

Cell-1 解压chm

In [ ]:
%cd /content/drive/MyDrive/Colab_Projects/revit-api-rag

# 创建目录
!mkdir -p data/raw/api_html
!mkdir -p data/raw/sdk_samples

# 解压 CHM 到 Colab 本地（不是 Drive，速度快很多）
!apt-get install -y p7zip-full -q
!7z x "/content/drive/MyDrive/Colab_Projects/revit_data/RevitAPI.chm" -o/tmp/chm_out -y

# 看看解出来什么
!ls /tmp/chm_out/

### Step 2b: Quality Agent — 低质量 API 数据修复（Gemini 审核 + Claude 重写）

In [ ]:
import importlib
import pipeline.api_parser.quality_agent as _qa_mod
importlib.reload(_qa_mod)
from pipeline.api_parser.quality_agent import run_quality_agent

import yaml
with open('/content/drive/MyDrive/Colab_Projects/revit-api-rag/config/config.yaml') as f:
    config = yaml.safe_load(f)

# ── 参数 ─────────────────────────────────────────
HTML_DIR    = '/content/api_html/'
# Stage-2 最多修复条数（调大会增加 Claude token 消耗）
MAX_STAGE2  = 3000
# ─────────────────────────────────────────────────

api_data_clean = run_quality_agent(
    api_data,
    config,
    html_dir=HTML_DIR,
    max_stage2=MAX_STAGE2,
    verbose=True,
)

# 存入 SQLite（覆盖旧数据）
!rm -f /content/drive/MyDrive/Colab_Projects/revit-api-rag/data/sqlite/revit_api.db
save_to_sqlite(api_data_clean, '/content/drive/MyDrive/Colab_Projects/revit-api-rag/data/sqlite/revit_api.db')
print(f'\nQuality Agent 修复后入库完成：{len(api_data_clean)} 条 ✅')


复制API HTML到项目目录

In [ ]:
%cd /content

# 1. 解压到 Colab 本地磁盘（不经过 Drive，非常快）
!mkdir -p /content/api_html
!7z x "/content/drive/MyDrive/Colab_Projects/revit_data/RevitAPI.chm" -o/content/chm_tmp -y
!mv /content/chm_tmp/html/* /content/api_html/
!rm -rf /content/chm_tmp

# 2. 验证
!find /content/api_html/ -name "*.htm*" | wc -l
print('API 解压到本地完成 ✅')

In [ ]:
# 3. 打包成一个 tar.gz（一个大文件写入 Drive 不会超时）
!tar -czf /content/drive/MyDrive/Colab_Projects/revit_data/api_html.tar.gz -C /content api_html/
print('API 打包完成 ✅')

API 打包完成 ✅


In [ ]:
# 4. SDK 解压到本地
!mkdir -p /content/sdk_samples
!unzip -o -q "/content/drive/MyDrive/Colab_Projects/revit_data/Samples.zip" -d /content/sdk_samples/

# 验证
!find /content/sdk_samples/ -name "*.cs" | wc -l
print('SDK 解压到本地完成 ✅')

1668
SDK 解压到本地完成 ✅


In [ ]:
# 5. SDK 打包
!tar -czf /content/drive/MyDrive/Colab_Projects/revit_data/sdk_samples.tar.gz -C /content sdk_samples/
print('SDK 打包完成 ✅')

SDK 打包完成 ✅


## 第二次运行从此处开始运行，首先检查是否有解压文件，没有解压则开始解压
Cell2 - Colab 重新连接，找到打包的压缩文件\

In [21]:
# 快速恢复数据（解压 tar.gz 到本地，几秒钟）
!tar -xzf /content/drive/MyDrive/Colab_Projects/revit_data/api_html.tar.gz -C /content/
!tar -xzf /content/drive/MyDrive/Colab_Projects/revit_data/sdk_samples.tar.gz -C /content/
print('数据恢复完成 ✅')

数据恢复完成 ✅


## Step 2: 解析 API 文档

In [ ]:
import sys
sys.path.insert(0,'/content/drive/MyDrive/Colab_Projects/revit-api-rag')

# 重新加载模块（因为之前 import 过旧版本）
import importlib
import pipeline.api_parser.parse_chm as parse_module
importlib.reload(parse_module)
from pipeline.api_parser.parse_chm import parse_all_api_html, save_to_sqlite

# 解析（用本地路径，快）
api_data = parse_all_api_html('/content/api_html/')

# 先看看前3条数据的质量
for item in api_data[:3]:
    print(f"名称: {item['name']}")
    print(f"命名空间: {item['namespace']}")
    print(f"描述: {item['info'][:100]}...")
    print(f"成员数: {len(item['members'].split(chr(10))) if item['members'] else 0}")
    print("---")

# 存入 SQLite
save_to_sqlite(api_data, '/content/drive/MyDrive/Colab_Projects/revit-api-rag/data/sqlite/revit_api.db')
print(f'\nAPI 解析完成：{len(api_data)} 条 ✅')

~删除原有的旧数据库数据，重新保存~

In [ ]:
!rm -f /content/drive/MyDrive/Colab_Projects/revit-api-rag/data/sqlite/revit_api.db

save_to_sqlite(api_data, '/content/drive/MyDrive/Colab_Projects/revit-api-rag/data/sqlite/revit_api.db')
print(f'\nAPI 解析完成：{len(api_data)} 条 ✅')

已保存 28863 条数据到 /content/drive/MyDrive/Colab_Projects/revit-api-rag/data/sqlite/revit_api.db

API 解析完成：28863 条 ✅


## Step 3: 解析 SDK 代码

In [ ]:
# 验证
!find /content/sdk_samples/ -name "*.cs" | wc -l
print('SDK 解压完成 ✅')

1668
SDK 解压完成 ✅


In [ ]:
from pipeline.sdk_parser.extract import extract_all_sdk_projects, save_to_sqlite

sdk_data = extract_all_sdk_projects('/content/sdk_samples/')

save_to_sqlite(sdk_data, '/content/drive/MyDrive/Colab_Projects/revit-api-rag/data/sqlite/revit_sdk.db')

print(f'SDK 解析完成：{len(sdk_data)} 条 ✅')

找到 1 个 SDK 项目
共提取 1668 个 .cs 文件
已保存 1668 条数据到 /content/drive/MyDrive/Colab_Projects/revit-api-rag/data/sqlite/revit_sdk.db
SDK 解析完成：1668 条 ✅


## Step 4: Embedding 向量化

设置APIKEY All From OpenRouter

In [ ]:
import os
from google.colab import userdata

os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
print('OpenRouter API Key 已设置 ✅')

OpenRouter API Key 已设置 ✅


In [ ]:
# 测试 embedding 是否正常
import importlib
import config as config_module
importlib.reload(config_module)
from config import load_config
from pipeline.embedder.providers import create_embedding

config = load_config('/content/drive/MyDrive/Colab_Projects/revit-api-rag/config/config.yaml')
embedder = create_embedding(config)

# 测试一条
test = embedder.embed_query("Create structural column in Revit")
print(f'Model: {embedder.model_name}')
print(f'Dimension: {len(test)}')
print(f'前5个值: {test[:5]}')
print('Embedding 测试成功 ✅')

Model: openai/text-embedding-3-large
Dimension: 3072
前5个值: [-0.0051480429247021675, -0.05087319016456604, -0.018109237775206566, 0.011834426783025265, 0.028094956651329994]
Embedding 测试成功 ✅


开始Embedding

In [ ]:
from config import load_config
from pipeline.embedder.embed import embed_api_data, embed_code_data

config = load_config('/content/drive/MyDrive/Colab_Projects/revit-api-rag/config/config.yaml')
version = config.get('revit_version', '2026')

print(f'Embedding provider: {config["embedding"]["provider"]}')
print(f'Revit version: {version}')
print('开始向量化...')

Embedding provider: openai
Revit version: 2026
开始向量化...


In [ ]:
base = '/content/drive/MyDrive/Colab_Projects/revit-api-rag'

# API 向量化
embed_api_data(
    config=config,
    api_db_path=f'{base}/data/sqlite/revit_api.db',
    chromadb_dir='/content/chromadb_api/',
)
print('API 向量化完成 ✅')

In [ ]:
# SDK 向量化
import importlib
import pipeline.embedder.embed as embed_module
importlib.reload(embed_module)
from pipeline.embedder.embed import embed_code_data

embed_code_data(
    config=config,
    sdk_db_path=f'{base}/data/sqlite/revit_sdk.db',
    chromadb_dir='/content/chromadb_code/',
)
print('SDK 向量化完成 ✅')

从 /content/drive/MyDrive/Colab_Projects/revit-api-rag/data/sqlite/revit_sdk.db 读取 0 条 SDK 代码数据
已写入 /content/chromadb_code/meta.json
Code 向量化完成，共 0 条，存入 /content/chromadb_code/
SDK 向量化完成 ✅


In [ ]:
# 完成后打包到 Drive
!tar -czf {base}/data/chromadb_code.tar.gz -C /content chromadb_code/
print('SDK 向量库已保存到 Drive ✅')

SDK 向量库已保存到 Drive ✅


打包回Drive

In [ ]:
# 打包存回 Drive
!tar -czf {base}/data/chromadb_api.tar.gz -C /content chromadb_api/
!tar -czf {base}/data/chromadb_code.tar.gz -C /content chromadb_code/
print('向量库已保存到 Drive ✅')

向量库已保存到 Drive ✅


## Step 5: 验证 & 下载

In [ ]:
import chromadb
import json

paths = {
    'api': '/content/chromadb_api/',
    'code': '/content/chromadb_code/',
}

for db_type, db_dir in paths.items():
    meta_path = f'{db_dir}/meta.json'
    try:
        with open(meta_path) as f:
            meta = json.load(f)
        print(f'\n{db_type.upper()} 向量库:')
        print(f'  Provider: {meta["embedding_provider"]}')
        print(f'  Model: {meta["embedding_model"]}')
        print(f'  Dimension: {meta["embedding_dimension"]}')
        print(f'  Records: {meta["record_count"]}')

        client = chromadb.PersistentClient(path=db_dir)
        for col in client.list_collections():
            print(f'  Collection: {col.name}, Count: {col.count()}')
    except FileNotFoundError:
        print(f'\n{db_type.upper()} 向量库: 未找到，需要重新生成')


API 向量库:
  Provider: openai
  Model: openai/text-embedding-3-large
  Dimension: 3072
  Records: 28863
  Collection: revit_api, Count: 28863

CODE 向量库:
  Provider: openai
  Model: openai/text-embedding-3-large
  Dimension: 3072
  Records: 1668
  Collection: revit_sdk, Count: 1668


In [ ]:
# 打包数据文件用于下载
!tar -czf revit_rag_data.tar.gz data/
print('数据已打包: revit_rag_data.tar.gz')
print(f'文件大小: {os.path.getsize("revit_rag_data.tar.gz") / 1024 / 1024:.1f} MB')

# 下载
from google.colab import files
files.download('revit_rag_data.tar.gz')

# 或者保存到 Google Drive
# !cp revit_rag_data.tar.gz /content/drive/MyDrive/

数据已打包: revit_rag_data.tar.gz
文件大小: 0.0 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 测试数据库db文件准确度

In [ ]:
# Cell 1: 加载向量库 + 配置
import chromadb
from pipeline.embedder.providers import create_embedding
from config import load_config

base = '/content/drive/MyDrive/Colab_Projects/revit-api-rag'
config = load_config(f'{base}/config/config.yaml')
embedder = create_embedding(config)

# 加载 ChromaDB
api_client = chromadb.PersistentClient(path='/content/chromadb_api/')
code_client = chromadb.PersistentClient(path='/content/chromadb_code/')
api_collection = api_client.get_collection("revit_api")
code_collection = code_client.get_collection("revit_sdk")

print(f"API 向量库: {api_collection.count()} 条")
print(f"Code 向量库: {code_collection.count()} 条")
print("加载完成 ✅")

In [ ]:
# Cell 2: 检索函数
def search_rag(query: str, api_top_k: int = 10, code_top_k: int = 5):
    """向量检索"""
    query_embedding = embedder.embed_query(query)

    api_results = api_collection.query(
        query_embeddings=[query_embedding],
        n_results=api_top_k,
    )

    code_results = code_collection.query(
        query_embeddings=[query_embedding],
        n_results=code_top_k,
    )

    return api_results, code_results


def format_results(api_results, code_results):
    """格式化检索结果"""
    print("=" * 60)
    print("📖 API 检索结果：")
    print("=" * 60)
    for i, (doc, meta, dist) in enumerate(zip(
        api_results['documents'][0],
        api_results['metadatas'][0],
        api_results['distances'][0]
    )):
        print(f"\n[{i+1}] 相似度: {1-dist:.4f}")
        print(f"    名称: {meta.get('name', '')}")
        print(f"    内容: {doc[:150]}...")

    print("\n" + "=" * 60)
    print("💻 代码检索结果：")
    print("=" * 60)
    for i, (doc, meta, dist) in enumerate(zip(
        code_results['documents'][0],
        code_results['metadatas'][0],
        code_results['distances'][0]
    )):
        print(f"\n[{i+1}] 相似度: {1-dist:.4f}")
        print(f"    项目: {meta.get('project', '')} / {meta.get('filename', '')}")
        print(f"    代码: {doc[:200]}...")

In [ ]:
# Cell 3: 测试检索
query = "创建结构柱"
api_results, code_results = search_rag(query)
format_results(api_results, code_results)

In [ ]:
# Cell 4: LLM 生成（通过 OpenRouter）
from openai import OpenAI
import os

llm_client = OpenAI(
    api_key=os.environ['OPENROUTER_API_KEY'],
    base_url="https://openrouter.ai/api/v1",
)

def generate_code(query: str, api_results, code_results, model: str = "google/gemini-2.5-flash"):
    """RAG 生成：检索结果 + LLM 生成代码"""

    # 拼接检索到的 API 参考
    api_context = "\n\n".join([
        f"API: {meta.get('name', '')}\n{doc[:300]}"
        for doc, meta in zip(api_results['documents'][0], api_results['metadatas'][0])
    ])

    # 拼接检索到的代码参考
    code_context = "\n\n".join([
        f"// File: {meta.get('filename', '')}\n{doc[:500]}"
        for doc, meta in zip(code_results['documents'][0], code_results['metadatas'][0])
    ])

    system_prompt = """You are a professional BIM engineer expert in Revit API.
You write C# code for Revit plugins following these standards:
1. Completeness: Include the entire process from start to finish
2. Professionalism: Correctly handle Revit element characteristics
3. Robustness: Include error handling and boundary condition checking
4. Scalability: Easy to extend
5. Best practice: Follow Revit API development specifications

Give the user a complete, working C# plugin code solution."""

    user_prompt = f"""User question: {query}

Revit API Reference:
{api_context}

Code Reference:
{code_context}

Based on the references above, generate a complete Revit C# plugin to solve the user's question.
Include all necessary using statements, the IExternalCommand class, and proper Transaction handling."""

    response = llm_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.3,
        max_tokens=4096,
    )

    return response.choices[0].message.content


# 运行完整 RAG 流程
query = "创建结构柱"
print(f"🔍 查询: {query}\n")

# 检索
api_results, code_results = search_rag(query, api_top_k=15, code_top_k=5)
print(f"检索到 API: {len(api_results['documents'][0])} 条, Code: {len(code_results['documents'][0])} 条")

# 生成
print("\n🤖 生成中...\n")
answer = generate_code(query, api_results, code_results)
print(answer)

## 完成！

下一步：
1. 将 `revit_rag_data.tar.gz` 上传到 GCP 服务器
2. 解压到项目的 `data/` 目录
3. 运行 `python -m server.app.main` 启动服务